# Week 3 — Chooser Option (BSM) 现跑通

本 notebook 目标：用 **Futu** 拉到的港股数据（`S0` + 历史价格估计 `sigma`），在 **BSM** 假设下用 **Monte Carlo** 方式把 chooser option 定价流程跑通。

不做论文对齐；只验证：数据可得 + 参数可配置 + 定价可复现。

## 1) 读取 Futu 输出数据

先运行脚本生成 CSV（示例）：

```bash
python src/scripts/futu_test.py --codes HK.00700 --start 2025-02-26 --end 2026-02-26 --calc-sigma
```

需要的文件：
- `output/futu/snapshot.csv`
- `output/futu/kline_HK_00700.csv`

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

root = Path('.').resolve()
snapshot_path = root / 'output' / 'futu' / 'snapshot.csv'
kline_path = root / 'output' / 'futu' / 'kline_HK_00700.csv'

assert snapshot_path.exists(), f"missing: {snapshot_path}"
assert kline_path.exists(), f"missing: {kline_path}"

snap = pd.read_csv(snapshot_path)
kline = pd.read_csv(kline_path)
display(snap.head(1))
display(kline.head(3))

S0 = float(snap.loc[0, 'last_price'])
print('S0 =', S0)

## 2) 用历史日线估计年化波动率 `sigma`

默认用最近 252 个对数日收益率，按 252 交易日年化。

In [ ]:
kline_sorted = kline.sort_values('time_key') if 'time_key' in kline.columns else kline
px = pd.to_numeric(kline_sorted['close'], errors='coerce').dropna()
rets = np.log(px).diff().dropna()

window = 252
trading_days = 252

rets_w = rets.tail(window)
sigma_daily = float(rets_w.std(ddof=1))
sigma = sigma_daily * math.sqrt(trading_days)

print('sigma_daily =', sigma_daily)
print('sigma_annualized =', sigma)

## 3) Chooser option（BSM）Monte Carlo 定价

Chooser 定义：在 `T1` 决定买入 **call 或 put（同 `K`、同到期 `T2`）** 中价值更高者，并在 `T2` 结算对应 payoff。

在 BSM 下，选择边界只取决于 `S(T1)`：


$$S^* = K \cdot \exp\left(-(r-q)(T2-T1)\right)$$

- 若 `S(T1) >= S*` 选择 call
- 否则选择 put

注意：港股标的是 HKD 计价；这里 `K=150` 仅用于流程跑通（真实定价要保证币种/单位一致）。

In [ ]:
# Parameters
K = 150.0
T1 = 0.5
T2 = 1.0
r = 0.0
q = 0.0

n_paths = 200_000
seed = 7

assert 0.0 < T1 < T2
assert sigma > 0

rng = np.random.default_rng(seed)

dt1 = T1
dt2 = T2 - T1

drift1 = (r - q - 0.5 * sigma**2) * dt1
drift2 = (r - q - 0.5 * sigma**2) * dt2
vol1 = sigma * math.sqrt(dt1)
vol2 = sigma * math.sqrt(dt2)

z1 = rng.standard_normal(n_paths)
z2 = rng.standard_normal(n_paths)

S_t1 = S0 * np.exp(drift1 + vol1 * z1)
S_t2 = S_t1 * np.exp(drift2 + vol2 * z2)

S_star = K * math.exp(-(r - q) * (T2 - T1))
choose_call = S_t1 >= S_star

payoff = np.where(choose_call, np.maximum(S_t2 - K, 0.0), np.maximum(K - S_t2, 0.0))
pv = math.exp(-r * T2) * payoff

price = float(pv.mean())
stderr = float(pv.std(ddof=1) / math.sqrt(n_paths))
ci95 = 1.96 * stderr

print('chooser_mc_price =', price)
print('stderr =', stderr)
print('ci95 =', (price - ci95, price + ci95))
print('p_choose_call =', float(choose_call.mean()))

## 4) 保存一次运行摘要

写到 `output/futu/chooser_notebook_summary.json`，方便后续写“验证报告”。

In [ ]:
out = {
    'ok': True,
    'asof': str(pd.Timestamp.now()),
    'inputs': {
        'S0': S0,
        'K': K,
        'T1': T1,
        'T2': T2,
        'r': r,
        'q': q,
        'sigma': sigma,
        'sigma_window': int(window),
        'trading_days': int(trading_days),
        'n_paths': int(n_paths),
        'seed': int(seed),
        'snapshot_csv': str(snapshot_path),
        'kline_csv': str(kline_path),
    },
    'result': {
        'price': price,
        'stderr': stderr,
        'ci95_low': price - ci95,
        'ci95_high': price + ci95,
        'p_choose_call': float(choose_call.mean()),
    },
    'notes': [
        'Not paper-matched; this is a run-through using Futu data + BSM Monte Carlo.',
        'Ensure K is in same currency/unit as S0 for real pricing.',
    ],
}

out_path = root / 'output' / 'futu' / 'chooser_notebook_summary.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('saved:', out_path)